In [ ]:
#@title Install Dependencies
%%capture
!pip install gradio openai -q

In [ ]:
#@title Set Your OpenAI API Key
import os
from getpass import getpass

# Enter your OpenAI API key when prompted
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
print("API key set successfully!")

In [ ]:
#@title LLM Output Evaluator App
"""
LLM Output Evaluator - Use AI to evaluate AI outputs
Demonstrates LLM-as-Judge pattern for non-technical PMs
"""

import gradio as gr
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

SAMPLE_OUTPUTS = {
    "Customer Service Response": {
        "prompt": "Customer asks: 'My order hasn't arrived and it's been 2 weeks. I'm very frustrated!'",
        "output": "I understand your frustration with the delayed order. Let me look into this right away. I'll check the tracking status and contact our shipping partner to locate your package. If we can't resolve this within 24 hours, I'll personally ensure you receive a full refund or expedited replacement. Is there anything else I can help you with while I investigate?"
    },
    "Technical Explanation": {
        "prompt": "Explain how a neural network learns, for a business audience",
        "output": "Think of a neural network like a new employee learning their job. At first, they make mistakes, but each time they get feedback, they adjust their approach. The network does this millions of times with training data, gradually improving its accuracy. The 'learning' is really just adjusting internal settings based on how wrong its predictions were - similar to how we learn from our errors."
    },
    "Product Description": {
        "prompt": "Write a product description for a smart water bottle that tracks hydration",
        "output": "Stay optimally hydrated with HydroTrack Pro. This intelligent water bottle monitors your daily intake and syncs with your phone to send gentle reminders when you need to drink. The LED ring glows when it's time to hydrate, and the app shows your weekly trends. BPA-free, dishwasher safe, 24-hour battery life. Your personal hydration coach, simplified."
    },
    "Code Review Comment": {
        "prompt": "Review this code: for i in range(len(items)): process(items[i])",
        "output": "Consider using a direct iteration instead: 'for item in items: process(item)'. This is more Pythonic, clearer to read, and slightly more efficient since it avoids index lookups. If you need the index, use enumerate(): 'for i, item in enumerate(items)'."
    }
}

EVALUATION_CRITERIA = {
    "Accuracy": "Information is factually correct and verifiable",
    "Relevance": "Response directly addresses the prompt/question",
    "Helpfulness": "Response provides actionable value to the user",
    "Clarity": "Response is easy to understand, well-structured",
    "Tone": "Tone is appropriate for the context and audience",
    "Completeness": "Response covers all necessary aspects"
}


def evaluate_with_llm(prompt, output, criteria_weights):
    """Use GPT-4o-mini to evaluate the output"""
    
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: Please set your OpenAI API key in the cell above.", "", ""
    
    selected_criteria = [c for c, w in criteria_weights.items() if w > 0]
    
    criteria_text = "\n".join([f"- {c}: {EVALUATION_CRITERIA[c]}" for c in selected_criteria])
    
    eval_prompt = f"""You are an expert evaluator assessing AI-generated content quality.

ORIGINAL PROMPT:
{prompt}

AI OUTPUT TO EVALUATE:
{output}

EVALUATION CRITERIA:
{criteria_text}

Please evaluate the output on each criterion using a 1-5 scale:
1 = Poor, 2 = Below Average, 3 = Average, 4 = Good, 5 = Excellent

For each criterion, provide:
1. A score (1-5)
2. A brief explanation (1-2 sentences)

Then provide:
- Overall score (weighted average)
- Key strengths (2-3 bullet points)
- Areas for improvement (2-3 bullet points)
- Overall assessment (2-3 sentences)

Format your response clearly with headers."""

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that evaluates AI outputs objectively and constructively."},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=0.3,
            max_tokens=800
        )
        
        evaluation = completion.choices[0].message.content
        
        human_note = """## Human Evaluation Comparison

For best results, compare this AI evaluation with human judgment:

1. **Does the AI score match your intuition?** If not, the criteria may need adjustment.
2. **Did the AI catch issues you missed?** LLM judges can be thorough.
3. **Did the AI miss obvious problems?** LLMs have biases (length, confidence).

**Remember:** LLM-as-Judge is a tool for scale, not a replacement for human oversight."""
        
        return evaluation, human_note, "Evaluation complete!"
        
    except Exception as e:
        return f"Error calling API: {str(e)}", "", "Evaluation failed"


def load_sample(sample_name):
    """Load a sample prompt and output"""
    if sample_name in SAMPLE_OUTPUTS:
        sample = SAMPLE_OUTPUTS[sample_name]
        return sample["prompt"], sample["output"]
    return "", ""


# Build Gradio interface
with gr.Blocks(title="LLM Output Evaluator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # LLM Output Evaluator
    
    Use AI to evaluate AI outputs - the LLM-as-Judge pattern.
    
    **For Product Managers:** This demonstrates automated quality assessment at scale.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")
            
            sample_dropdown = gr.Dropdown(
                choices=list(SAMPLE_OUTPUTS.keys()),
                label="Load Sample",
                value="Customer Service Response"
            )
            
            prompt_input = gr.Textbox(
                label="Original Prompt",
                placeholder="What was the AI asked to do?",
                lines=3
            )
            
            output_input = gr.Textbox(
                label="AI Output to Evaluate",
                placeholder="Paste the AI's response here",
                lines=6
            )
            
            gr.Markdown("### Evaluation Criteria Weights")
            
            criteria_sliders = {}
            for criterion in EVALUATION_CRITERIA:
                criteria_sliders[criterion] = gr.Slider(
                    minimum=0, maximum=1, value=1, step=0.1,
                    label=f"{criterion}"
                )
            
            evaluate_btn = gr.Button("Evaluate with AI Judge", variant="primary")
        
        with gr.Column(scale=2):
            gr.Markdown("### AI Judge Evaluation")
            evaluation_output = gr.Markdown()
            human_comparison = gr.Markdown()
            status = gr.Textbox(label="Status", interactive=False)
    
    gr.Markdown("""
    ---
    ### PM Guide: LLM-as-Judge
    
    **When to use:** Scaling evaluation to thousands of samples
    
    **Known biases:**
    - Length bias: Longer responses often rated higher
    - Position bias: First option in comparisons preferred
    - Self-preference: Models prefer their own style
    
    **Best practices:**
    - Validate against human judgments on a sample set
    - Use specific, measurable criteria
    - Consider using multiple judge models
    """)
    
    # Event handlers
    sample_dropdown.change(
        fn=load_sample,
        inputs=[sample_dropdown],
        outputs=[prompt_input, output_input]
    )
    
    def run_evaluation(prompt, output, *criteria_values):
        criteria_weights = dict(zip(EVALUATION_CRITERIA.keys(), criteria_values))
        return evaluate_with_llm(prompt, output, criteria_weights)
    
    evaluate_btn.click(
        fn=run_evaluation,
        inputs=[prompt_input, output_input] + list(criteria_sliders.values()),
        outputs=[evaluation_output, human_comparison, status]
    )
    
    # Load initial sample
    demo.load(
        fn=load_sample,
        inputs=[sample_dropdown],
        outputs=[prompt_input, output_input]
    )

In [ ]:
#@title Launch App - Copy the gradio.live URL below
demo.launch(share=True)